<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Launch Sites Locations Analysis with Folium**


Estimated time needed: **40** minutes


The launch success rate may depend on many factors such as payload mass, orbit type, and so on. It may also depend on the location and proximities of a launch site, i.e., the initial position of rocket trajectories. Finding an optimal location for building a launch site certainly involves many factors and hopefully we could discover some of the factors by analyzing the existing launch site locations.


In the previous exploratory data analysis labs, you have visualized the SpaceX launch dataset using `matplotlib` and `seaborn` and discovered some preliminary correlations between the launch site and success rates. In this lab, you will be performing more interactive visual analytics using `Folium`.


## Objectives


This lab contains the following tasks:
- **TASK 1:** Mark all launch sites on a map
- **TASK 2:** Mark the success/failed launches for each site on the map
- **TASK 3:** Calculate the distances between a launch site to its proximities

After completed the above tasks, you should be able to find some geographical patterns about launch sites.


Let's first import required Python packages for this lab:


In [1]:
# The required packages are imported below.
# No installation command is needed when they are already available.


In [2]:
%pip install folium
from io import StringIO
from pathlib import Path
from math import sin, cos, sqrt, atan2, radians

import folium
import pandas as pd
import requests


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: C:\Users\18201791\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
from folium.features import DivIcon
from folium.plugins import MarkerCluster, MousePosition

pd.set_option("display.max_columns", None)


If you need to refresh your memory about folium, you may download and refer to this previous folium lab:


[Generating Maps with Python](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/DV0101EN-3-5-1-Generating-Maps-in-Python-py-v2.0.ipynb)


## Task 1: Mark all launch sites on a map


First, let's try to add each site's location on a map using site's latitude and longitude coordinates


The following dataset with the name `spacex_launch_geo.csv` is an augmented dataset with latitude and longitude added for each site. 


In [4]:
# Load launch records and attach official launch-site coordinates.
# The local dataset_part_2.csv from the previous notebook is preferred.

local_candidates = [
    Path("dataset_part_2.csv"),
    Path("./dataset_part_2.csv"),
]

dataset_url = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_2.csv"
)

spacex_source = None

for candidate in local_candidates:
    if candidate.exists():
        spacex_source = pd.read_csv(candidate)
        print(f"Loaded local dataset: {candidate.resolve()}")
        break

if spacex_source is None:
    try:
        response = requests.get(dataset_url, timeout=60)
        response.raise_for_status()
        spacex_source = pd.read_csv(StringIO(response.text))
        print("Downloaded the official IBM dataset_part_2.csv.")
    except requests.RequestException as error:
        raise RuntimeError(
            "The launch dataset could not be loaded. Place dataset_part_2.csv "
            "in the same folder as this notebook and run the cell again."
        ) from error

# Coordinates used in IBM's SpaceX launch-site mapping lab.
site_coordinates = {
    "CCAFS LC-40": (28.562302, -80.577356),
    "CCAFS LC 40": (28.562302, -80.577356),

    "CCAFS SLC-40": (28.563197, -80.576820),
    "CCAFS SLC 40": (28.563197, -80.576820),

    "KSC LC-39A": (28.573255, -80.646895),
    "KSC LC 39A": (28.573255, -80.646895),

    "VAFB SLC-4E": (34.632834, -120.610745),
    "VAFB SLC 4E": (34.632834, -120.610745),
}

required_columns = {"LaunchSite", "Class"}
missing_columns = required_columns.difference(spacex_source.columns)

if missing_columns:
    raise ValueError(
        f"dataset_part_2.csv is missing required columns: "
        f"{sorted(missing_columns)}"
    )

spacex_df = spacex_source[["LaunchSite", "Class"]].copy()
spacex_df = spacex_df.rename(
    columns={
        "LaunchSite": "Launch Site",
        "Class": "class",
    }
)

spacex_df["Lat"] = spacex_df["Launch Site"].map(
    lambda site: site_coordinates.get(site, (None, None))[0]
)
spacex_df["Long"] = spacex_df["Launch Site"].map(
    lambda site: site_coordinates.get(site, (None, None))[1]
)

unknown_sites = sorted(
    spacex_df.loc[
        spacex_df[["Lat", "Long"]].isna().any(axis=1),
        "Launch Site"
    ].dropna().unique()
)

if unknown_sites:
    raise ValueError(
        "Coordinates are unavailable for these launch sites: "
        f"{unknown_sites}"
    )

spacex_df["class"] = pd.to_numeric(
    spacex_df["class"],
    errors="raise"
).astype(int)

print(f"Launch records loaded: {len(spacex_df)}")
spacex_df.head()


Loaded local dataset: C:\Users\18201791\Desktop\HPV_Project_GitHub\IBM-Applied-Data-Science-Capstone\dataset_part_2.csv
Launch records loaded: 90


,Launch Site,class,Lat,Long
0,CCAFS SLC 40,0,28.563197,-80.576820
1,CCAFS SLC 40,0,28.563197,-80.576820
2,CCAFS SLC 40,0,28.563197,-80.576820
3,VAFB SLC 4E,0,34.632834,-120.610745
4,CCAFS SLC 40,0,28.563197,-80.576820


Now, you can take a look at what are the coordinates for each site.


In [5]:
# Select relevant columns and obtain one coordinate row per launch site.
spacex_df = spacex_df[
    ["Launch Site", "Lat", "Long", "class"]
].copy()

launch_sites_df = (
    spacex_df.groupby("Launch Site", as_index=False)
             .agg(
                 Lat=("Lat", "first"),
                 Long=("Long", "first"),
                 Launches=("class", "size"),
                 Success_Rate=("class", "mean")
             )
)

launch_sites_df["Success_Rate"] = (
    launch_sites_df["Success_Rate"].round(3)
)

launch_sites_df


,Launch Site,Lat,Long,Launches,Success_Rate
0,CCAFS SLC 40,28.563197,-80.576820,55,0.600
1,KSC LC 39A,28.573255,-80.646895,22,0.773
2,VAFB SLC 4E,34.632834,-120.610745,13,0.769


Above coordinates are just plain numbers that can not give you any intuitive insights about where are those launch sites. If you are very good at geography, you can interpret those numbers directly in your mind. If not, that's fine too. Let's visualize those locations by pinning them on a map.


We first need to create a folium `Map` object, with an initial center location to be NASA Johnson Space Center at Houston, Texas.


In [6]:
# Start location: NASA Johnson Space Center, Houston, Texas.
nasa_coordinate = [29.559684888503615, -95.0830971930759]

site_map = folium.Map(
    location=nasa_coordinate,
    zoom_start=4,
    tiles="OpenStreetMap"
)

site_map


We could use `folium.Circle` to add a highlighted circle area with a text label on a specific coordinate. For example, 


In [7]:
# Add NASA Johnson Space Center as a reference point.
circle = folium.Circle(
    location=nasa_coordinate,
    radius=1000,
    color="#d35400",
    fill=True,
    fill_opacity=0.35,
    popup="NASA Johnson Space Center",
)

marker = folium.map.Marker(
    nasa_coordinate,
    icon=DivIcon(
        icon_size=(180, 24),
        icon_anchor=(0, 0),
        html=(
            '<div style="font-size:12px;color:#d35400;">'
            '<b>NASA Johnson Space Center</b></div>'
        ),
    ),
)

site_map.add_child(circle)
site_map.add_child(marker)
site_map


and you should find a small yellow circle near the city of Houston and you can zoom-in to see a larger circle. 


Now, let's add a circle for each launch site in data frame `launch_sites`


_TODO:_  Create and add `folium.Circle` and `folium.Marker` for each launch site on the site map


An example of folium.Circle:


`folium.Circle(coordinate, radius=1000, color='#000000', fill=True).add_child(folium.Popup(...))`


An example of folium.Marker:


`folium.map.Marker(coordinate, icon=DivIcon(icon_size=(20,20),icon_anchor=(0,0), html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'label', ))`


In [8]:
# TASK 1: Mark every launch site with a circle and text label.
site_map = folium.Map(
    location=[31.0, -98.0],
    zoom_start=4,
    tiles="OpenStreetMap"
)

for _, site in launch_sites_df.iterrows():
    coordinate = [site["Lat"], site["Long"]]

    popup_text = (
        f"{site['Launch Site']}<br>"
        f"Launches: {int(site['Launches'])}<br>"
        f"Success rate: {site['Success_Rate']:.1%}"
    )

    folium.Circle(
        location=coordinate,
        radius=1000,
        color="#000000",
        fill=True,
        fill_color="#3186cc",
        fill_opacity=0.35,
        popup=folium.Popup(popup_text, max_width=250),
    ).add_to(site_map)

    folium.Marker(
        location=coordinate,
        icon=DivIcon(
            icon_size=(180, 30),
            icon_anchor=(0, 0),
            html=(
                '<div style="font-size:11px;color:#d35400;">'
                f"<b>{site['Launch Site']}</b></div>"
            ),
        ),
    ).add_to(site_map)

print("All launch sites were marked successfully.")
site_map


All launch sites were marked successfully.


The generated map with marked launch sites should look similar to the following:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_markers.png">
</center>


Now, you can explore the map by zoom-in/out the marked areas
, and try to answer the following questions:
- Are all launch sites in proximity to the Equator line?
- Are all launch sites in very close proximity to the coast?

Also please try to explain your findings.


# Task 2: Mark the success/failed launches for each site on the map


Next, let's try to enhance the map by adding the launch outcomes for each site, and see which sites have high success rates.
Recall that data frame spacex_df has detailed launch records, and the `class` column indicates if this launch was successful or not


In [9]:
spacex_df.tail(10)

,Launch Site,Lat,Long,class
80,CCAFS SLC 40,28.563197,-80.576820,1
81,CCAFS SLC 40,28.563197,-80.576820,1
82,CCAFS SLC 40,28.563197,-80.576820,1
83,CCAFS SLC 40,28.563197,-80.576820,1
84,CCAFS SLC 40,28.563197,-80.576820,1
85,KSC LC 39A,28.573255,-80.646895,1
86,KSC LC 39A,28.573255,-80.646895,1
87,KSC LC 39A,28.573255,-80.646895,1
88,CCAFS SLC 40,28.563197,-80.576820,1
89,CCAFS SLC 40,28.563197,-80.576820,1


Next, let's create markers for all launch records. 
If a launch was successful `(class=1)`, then we use a green marker and if a launch was failed, we use a red marker `(class=0)`


Note that a launch only happens in one of the four launch sites, which means many launch records will have the exact same coordinate. Marker clusters can be a good way to simplify a map containing many markers having the same coordinate.


Let's first create a `MarkerCluster` object


In [10]:
marker_cluster = MarkerCluster(
    name="Launch outcomes"
)

print("Marker cluster created.")


Marker cluster created.


_TODO:_ Create a new column in `launch_sites` dataframe called `marker_color` to store the marker colors based on the `class` value


In [11]:
# TASK 2: Create marker colours from the binary landing class.
spacex_df["marker_color"] = spacex_df["class"].map(
    {1: "green", 0: "red"}
)

if spacex_df["marker_color"].isna().any():
    invalid_classes = sorted(
        spacex_df.loc[
            spacex_df["marker_color"].isna(),
            "class"
        ].unique()
    )
    raise ValueError(
        f"Unexpected class values were found: {invalid_classes}"
    )

spacex_df[
    ["Launch Site", "class", "marker_color"]
].head(10)


,Launch Site,class,marker_color
0,CCAFS SLC 40,0,red
1,CCAFS SLC 40,0,red
2,CCAFS SLC 40,0,red
3,VAFB SLC 4E,0,red
4,CCAFS SLC 40,0,red
5,CCAFS SLC 40,0,red
6,CCAFS SLC 40,1,green
7,CCAFS SLC 40,1,green
8,CCAFS SLC 40,0,red
9,CCAFS SLC 40,0,red


In [12]:
# Function retained for the task specification.
def assign_marker_color(launch_outcome):
    if int(launch_outcome) == 1:
        return "green"
    return "red"


# Confirm the function produces the same result.
spacex_df["marker_color"] = (
    spacex_df["class"].apply(assign_marker_color)
)

spacex_df["marker_color"].value_counts()


marker_color
green    60
red      30
Name: count, dtype: int64

_TODO:_ For each launch result in `spacex_df` data frame, add a `folium.Marker` to `marker_cluster`


In [13]:
# Add one outcome marker for every launch record.
site_map.add_child(marker_cluster)

for _, launch in spacex_df.iterrows():
    outcome_text = (
        "Successful landing"
        if launch["class"] == 1
        else "Unsuccessful landing"
    )

    folium.Marker(
        location=[launch["Lat"], launch["Long"]],
        popup=(
            f"{launch['Launch Site']}<br>"
            f"{outcome_text}"
        ),
        icon=folium.Icon(
            color=launch["marker_color"],
            icon="info-sign",
        ),
    ).add_to(marker_cluster)

folium.LayerControl().add_to(site_map)

print("Success and failure markers were added successfully.")
site_map


Success and failure markers were added successfully.


Your updated map may look like the following screenshots:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster.png">
</center>


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster_zoomed.png">
</center>


From the color-labeled markers in marker clusters, you should be able to easily identify which launch sites have relatively high success rates.


# TASK 3: Calculate the distances between a launch site to its proximities


Next, we need to explore and analyze the proximities of launch sites.


Let's first add a `MousePosition` on the map to get coordinate for a mouse over a point on the map. As such, while you are exploring the map, you can easily find the coordinates of any points of interests (such as railway)


In [14]:
# Add a mouse-position coordinate display.
formatter = "function(num) {return L.Util.formatNum(num, 5);};"

mouse_position = MousePosition(
    position="topright",
    separator=" | ",
    empty_string="NaN",
    lng_first=False,
    num_digits=20,
    prefix="Coordinates:",
    lat_formatter=formatter,
    lng_formatter=formatter,
)

site_map.add_child(mouse_position)
site_map


Now zoom in to a launch site and explore its proximity to see if you can easily find any railway, highway, coastline, etc. Move your mouse to these points and mark down their coordinates (shown on the top-left) in order to the distance to the launch site.


You can calculate the distance between two points on the map based on their `Lat` and `Long` values using the following method:


In [15]:
def calculate_distance(lat1, lon1, lat2, lon2):
    """Calculate great-circle distance between two coordinates in kilometres."""
    earth_radius_km = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = (
        sin(dlat / 2) ** 2
        + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    )
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    return earth_radius_km * c


print(
    "Distance-function test:",
    round(calculate_distance(0, 0, 0, 1), 2),
    "km"
)


Distance-function test: 111.23 km


_TODO:_ Mark down a point on the closest coastline using MousePosition and calculate the distance between the coastline point and the launch site.


In [16]:
# TASK 3: Calculate the distance from KSC LC-39A to a nearby coastline point.
launch_site_name = "KSC LC-39A"
launch_site_lat, launch_site_lon = site_coordinates[launch_site_name]

# Approximate nearby Atlantic coastline coordinate.
coastline_lat = 28.61267
coastline_lon = -80.59721

distance_coastline = calculate_distance(
    launch_site_lat,
    launch_site_lon,
    coastline_lat,
    coastline_lon,
)

print(
    f"Distance from {launch_site_name} to the selected coastline: "
    f"{distance_coastline:.2f} km"
)


Distance from KSC LC-39A to the selected coastline: 6.54 km


_TODO:_ After obtained its coordinate, create a `folium.Marker` to show the distance


In [17]:
# Add a marker at the selected coastline point.
distance_marker = folium.Marker(
    location=[coastline_lat, coastline_lon],
    popup=(
        f"Closest selected coastline point<br>"
        f"Distance: {distance_coastline:.2f} km"
    ),
    icon=DivIcon(
        icon_size=(170, 36),
        icon_anchor=(0, 0),
        html=(
            '<div style="font-size:11px;color:#1f618d;">'
            f"<b>{distance_coastline:.2f} km to coastline</b>"
            "</div>"
        ),
    ),
)

site_map.add_child(distance_marker)
site_map


_TODO:_ Draw a `PolyLine` between a launch site to the selected coastline point


In [18]:
# Draw a line between KSC LC-39A and the selected coastline.
coastline_coordinates = [
    [launch_site_lat, launch_site_lon],
    [coastline_lat, coastline_lon],
]

coastline_line = folium.PolyLine(
    locations=coastline_coordinates,
    weight=3,
    opacity=0.8,
    tooltip=f"Coastline distance: {distance_coastline:.2f} km",
)

site_map.add_child(coastline_line)
site_map


Your updated map with distance line should look like the following screenshot:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_distance.png">
</center>


_TODO:_ Similarly, you can draw a line betwee a launch site to its closest city, railway, highway, etc. You need to use `MousePosition` to find the their coordinates on the map first


A railway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/railway.png">
</center>


A highway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/highway.png">
</center>


A city map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/city.png">
</center>


In [19]:
# Mark selected nearby transport and population features.
nearby_points = {
    "Closest selected city area": (28.4013, -80.6043),
    "Closest selected highway point": (28.5721, -80.6815),
    "Closest selected railway point": (28.5728, -80.6548),
}

proximity_results = []

for label, (point_lat, point_lon) in nearby_points.items():
    distance_km = calculate_distance(
        launch_site_lat,
        launch_site_lon,
        point_lat,
        point_lon,
    )

    proximity_results.append(
        {
            "Feature": label,
            "Distance_km": round(distance_km, 2),
            "Latitude": point_lat,
            "Longitude": point_lon,
        }
    )

    folium.Marker(
        location=[point_lat, point_lon],
        popup=f"{label}<br>{distance_km:.2f} km",
        icon=folium.Icon(color="blue", icon="info-sign"),
    ).add_to(site_map)

    folium.PolyLine(
        locations=[
            [launch_site_lat, launch_site_lon],
            [point_lat, point_lon],
        ],
        weight=2,
        opacity=0.7,
        tooltip=f"{label}: {distance_km:.2f} km",
    ).add_to(site_map)

proximity_df = pd.DataFrame(proximity_results)
proximity_df


,Feature,Distance_km,Latitude,Longitude
0,Closest selected city area,19.57,28.4013,-80.6043
1,Closest selected highway point,3.38,28.5721,-80.6815
2,Closest selected railway point,0.77,28.5728,-80.6548


In [20]:
# Save the completed interactive map as an HTML file.
map_output = "spacex_launch_site_analysis.html"
site_map.save(map_output)

print(f"Saved interactive map: {map_output}")
site_map


Saved interactive map: spacex_launch_site_analysis.html


In [21]:
# Final validation.
required_map_columns = {
    "Launch Site", "Lat", "Long", "class", "marker_color"
}

missing_map_columns = required_map_columns.difference(spacex_df.columns)

if missing_map_columns:
    raise ValueError(
        f"Missing required mapping columns: {sorted(missing_map_columns)}"
    )

if len(launch_sites_df) != 4:
    print(
        "Note: the dataset contains",
        len(launch_sites_df),
        "unique launch sites."
    )

if not Path("spacex_launch_site_analysis.html").exists():
    raise FileNotFoundError(
        "The interactive HTML map was not created."
    )

print("All Folium tasks and final validation completed successfully.")


Note: the dataset contains 3 unique launch sites.
All Folium tasks and final validation completed successfully.


After you plot distance lines to the proximities, you can answer the following questions easily:
- Are launch sites in close proximity to railways?
- Are launch sites in close proximity to highways?
- Are launch sites in close proximity to coastline?
- Do launch sites keep certain distance away from cities?

Also please try to explain your findings.


# Next Steps:

Now you have discovered many interesting insights related to the launch sites' location using folium, in a very interactive way. Next, you will need to build a dashboard using Ploty Dash on detailed launch records.


## Authors


[Yan Luo](https://www.linkedin.com/in/yan-luo-96288783/)


### Other Contributors


Joseph Santarcangelo


## Change Log


|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|
|2021-05-26|1.0|Yan|Created the initial version|


Copyright © 2021 IBM Corporation. All rights reserved.
